In [1]:
import pandas as pd
import numpy as np
import math
from pyproj import Transformer
from shapely.geometry import Point, LineString
import plotly.express as px

from datetime import time, datetime, timezone
import os
from pathlib import Path

In [10]:
import duckdb
from pathlib import Path

PARQ_BASE = Path('parquet')  # change if your parquet lives elsewhere
con = duckdb.connect()

def create_union_view(table_name: str, cast_cols: dict[str, str] | None = None):
    cast_cols = cast_cols or {}
    folder = PARQ_BASE / table_name
    if not folder.exists():
        raise FileNotFoundError(f'No parquet folder found at {folder}')
    path = (folder / '*.parquet').as_posix()
    base = f"read_parquet('{path}', union_by_name=True)"
    if cast_cols:
        cols = ', '.join(cast_cols.keys())
        casts = ', '.join([f"CAST({col} AS {typ}) AS {col}" for col, typ in cast_cols.items()])
        sql = f"SELECT * EXCLUDE ({cols}), {casts} FROM {base}"
    else:
        sql = f"SELECT * FROM {base}"
    con.execute(f"CREATE OR REPLACE TEMP VIEW {table_name} AS {sql}")

In [ ]:

# Register tables; cast service_id to VARCHAR to avoid mixed-type errors
create_union_view('dim_trips', {'service_id': 'VARCHAR'})
create_union_view('dim_routes')
create_union_view('calendar_base', {'service_id': 'VARCHAR'})
create_union_view('dim_stops')
create_union_view('fact_stop_events')

In [3]:
stops = con.execute('SELECT * FROM dim_stops').fetchdf()
routes = con.execute('SELECT * FROM dim_routes').fetchdf()
trips = con.execute('SELECT * FROM dim_trips').fetchdf()
facts = con.execute('SELECT * FROM fact_stop_events').fetchdf()

In [4]:
directions = trips[['route_id','direction_id','trip_headsign','feed_id','service_id']].drop_duplicates().sort_values('route_id')
directions = directions[directions['feed_id'] != 'miami-dade']

df = pd.merge(facts[['route_id', 'direction_id', 'service_id', 'stop_id', 'stop_sequence','arrival_sec', 'trip_id', 'feed_id']], 
              stops[['stop_id','stop_name','lat','lon','feed_id']], 
              on=['stop_id','feed_id',])
df = pd.merge(df, directions, on=['route_id','direction_id','feed_id','service_id'])
df = df.sort_values(["feed_id", "trip_id", "stop_sequence"])
df

,route_id,direction_id,service_id,stop_id,stop_sequence,arrival_sec,trip_id,feed_id,stop_name,lat,lon,trip_headsign
590164,BX41,1,GH_A6-Saturday,102793,1,0,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WHITE PLAINS RD/EAST GUN HILL RD,40.877829,-73.866304,THE HUB 150 ST via WEBSTER
590165,BX41,1,GH_A6-Saturday,102795,2,104,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WEBSTER AV/EAST GUN HILL RD,40.878196,-73.871756,THE HUB 150 ST via WEBSTER
590166,BX41,1,GH_A6-Saturday,102796,3,238,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WEBSTER AV/E 205 ST,40.872726,-73.875240,THE HUB 150 ST via WEBSTER
590167,BX41,1,GH_A6-Saturday,102797,4,292,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WEBSTER AV/E 204 ST,40.870873,-73.877332,THE HUB 150 ST via WEBSTER
590168,BX41,1,GH_A6-Saturday,102798,5,349,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WEBSTER AV/EAST MOSHOLU PKWY NORTH,40.869420,-73.880178,THE HUB 150 ST via WEBSTER
...,...,...,...,...,...,...,...,...,...,...,...,...
9703215,S59,1,YU_H6-Weekday,200475,50,93591,YU_H6-Weekday-152500_MISC_903,mta-nyct-bus-si,RICHMOND AV/SHIRLEY AV,40.536964,-74.158390,HYLAN BL
9703216,S59,1,YU_H6-Weekday,200476,51,93634,YU_H6-Weekday-152500_MISC_903,mta-nyct-bus-si,RICHMOND AV/KING ST,40.535112,-74.155659,TOTTENVILLE
9703217,S59,1,YU_H6-Weekday,200476,51,93634,YU_H6-Weekday-152500_MISC_903,mta-nyct-bus-si,RICHMOND AV/KING ST,40.535112,-74.155659,HYLAN BL
9703218,S59,1,YU_H6-Weekday,202856,52,93660,YU_H6-Weekday-152500_MISC_903,mta-nyct-bus-si,HYLAN BLVD/RICHMOND AV,40.534119,-74.154156,TOTTENVILLE


In [5]:
# facts has: feed_id, route_id, direction_id, service_id, trip_id, stop_id, stop_sequence
facts_sorted = df.sort_values(
    ["feed_id", "route_id",'trip_headsign', "direction_id", "service_id", "trip_id", "stop_sequence"]
)

# build a stop pattern per trip
patterns = (facts_sorted
    .groupby(["feed_id", "route_id", 'trip_headsign',"direction_id", "service_id", "trip_id"])["stop_id"]
    .apply(tuple)
    .reset_index(name="stop_pattern")
)

# count distinct patterns per group
pattern_counts = (patterns
    .groupby(["feed_id", "route_id",'trip_headsign', "direction_id", "service_id"])["stop_pattern"]
    .nunique()
    .reset_index(name="distinct_patterns")
)

# show groups with multiple patterns
multi = pattern_counts[pattern_counts["distinct_patterns"] > 1].sort_values(
    "distinct_patterns", ascending=False
)

multi.sort_values('route_id').head(50)

,feed_id,route_id,trip_headsign,direction_id,service_id,distinct_patterns
640,mta-nyct-bus-brooklyn,B1,OCEAN PKY,0,UP_A6-Weekday-SDon,5
633,mta-nyct-bus-brooklyn,B1,BAY RIDGE 4 AV,0,UP_A6-Weekday-SDon,5
638,mta-nyct-bus-brooklyn,B1,MANHATTAN BEACH KINGSBORO CC,1,UP_A6-Weekday-SDon,3
641,mta-nyct-bus-brooklyn,B1,STILLWELL AV 86 ST,0,UP_A6-Weekday-SDon,5
1523,mta-nyct-bus-busco,B100,MILL BASIN E. 66 ST,0,SCPA6-SC_A6-Weekday-01-SDon,4
1518,mta-nyct-bus-busco,B100,MIDWOOD KINGS HWY STA,1,SCPA6-SC_A6-Weekday-01-SDon,3
1514,mta-nyct-bus-busco,B100,BEDFORD AV,0,SCPA6-SC_A6-Weekday-01-SDon,4
1531,mta-nyct-bus-busco,B103,LIMITED CANARSIE - WILLIAMS AVE via AVENUE H v...,0,SCPA6-SC_A6-Sunday-01,2
1532,mta-nyct-bus-busco,B103,LIMITED CANARSIE - WILLIAMS AVE via AVENUE H v...,0,SCPA6-SC_A6-Weekday-01,2
1533,mta-nyct-bus-busco,B103,LIMITED CANARSIE - WILLIAMS AVE via AVENUE H v...,0,SCPA6-SC_A6-Weekday-01-SDon,2


In [5]:
def bearing_deg(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    dlon = np.radians(lon2 - lon1)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(y, x)) + 360.0) % 360.0

In [ ]:
# prev/next stop coords per trip
df["prev_lat"] = df.groupby(["feed_id","trip_id"])["lat"].shift(1)
df["prev_lon"] = df.groupby(["feed_id","trip_id"])["lon"].shift(1)
df["next_lat"] = df.groupby(["feed_id","trip_id"])["lat"].shift(-1)
df["next_lon"] = df.groupby(["feed_id","trip_id"])["lon"].shift(-1)

# compute three bearings when possible
df["bearing_prev_cur"] = bearing_deg(df["prev_lat"], df["prev_lon"], df["lat"], df["lon"])
df["bearing_cur_next"] = bearing_deg(df["lat"], df["lon"], df["next_lat"], df["next_lon"])
df["bearing_prev_next"] = bearing_deg(df["prev_lat"], df["prev_lon"], df["next_lat"], df["next_lon"])

df.head()

,route_id,direction_id,service_id,stop_id,stop_sequence,arrival_sec,trip_id,feed_id,stop_name,lat,lon,trip_headsign,prev_lat,prev_lon,next_lat,next_lon,bearing_prev_cur,bearing_cur_next,bearing_prev_next
590164,BX41,1,GH_A6-Saturday,102793,1,0,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WHITE PLAINS RD/EAST GUN HILL RD,40.877829,-73.866304,THE HUB 150 ST via WEBSTER,NaN,NaN,40.878196,-73.871756,NaN,275.089319,NaN
590165,BX41,1,GH_A6-Saturday,102795,2,104,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WEBSTER AV/EAST GUN HILL RD,40.878196,-73.871756,THE HUB 150 ST via WEBSTER,40.877829,-73.866304,40.872726,-73.875240,275.089319,205.716761,232.941517
590166,BX41,1,GH_A6-Saturday,102796,3,238,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WEBSTER AV/E 205 ST,40.872726,-73.875240,THE HUB 150 ST via WEBSTER,40.878196,-73.871756,40.870873,-73.877332,205.716761,220.488309,209.933268
590167,BX41,1,GH_A6-Saturday,102797,4,292,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WEBSTER AV/E 204 ST,40.870873,-73.877332,THE HUB 150 ST via WEBSTER,40.872726,-73.875240,40.869420,-73.880178,220.488309,235.975865,228.480933
590168,BX41,1,GH_A6-Saturday,102798,5,349,GH_A6-Saturday-000000_BX41_901,mta-nyct-bus-bronx,WEBSTER AV/EAST MOSHOLU PKWY NORTH,40.869420,-73.880178,THE HUB 150 ST via WEBSTER,40.870873,-73.877332,40.867348,-73.883511,235.975865,230.578456,232.971187


### Calculate near/far by distance

In [17]:
def filter_within_radius(df, lat_col, lon_col, center_lat, center_lon, radius_ft):
    radius_m = radius_ft * 0.3048
    lat = np.radians(df[lat_col].to_numpy())
    lon = np.radians(df[lon_col].to_numpy())
    clat = np.radians(center_lat)
    clon = np.radians(center_lon)

    dlat = lat - clat
    dlon = lon - clon
    a = np.sin(dlat/2)**2 + np.cos(clat) * np.cos(lat) * np.sin(dlon/2)**2
    dist_m = 2 * 6371000 * np.arcsin(np.sqrt(a))

    return df[dist_m <= radius_m].copy()

In [25]:
def calculate_bus_stop_side(df, intersection_lat, intersection_lon):
    """
    Calculate if each bus stop is on the near-side or far-side of an intersection.
    
    Parameters:
    -----------
    df : pandas DataFrame
        DataFrame with columns: lat, lon, prev_lat, prev_lon, next_lat, next_lon
    intersection_lat : float
        Latitude of the intersection point
    intersection_lon : float
        Longitude of the intersection point
    
    Returns:
    --------
    pandas DataFrame
        Original DataFrame with added columns:
        - dist_to_prev_current: distance from intersection to prev→current segment (in feet)
        - dist_to_current_next: distance from intersection to current→next segment (in feet)
        - stop_side: 'near-side' or 'far-side'
    """
    # Use EPSG:2263 (NY State Plane Long Island, US Survey Feet)
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:2263", always_xy=True)
    
    # Convert intersection point to projected coordinates
    intersection_x, intersection_y = transformer.transform(intersection_lon, intersection_lat)
    intersection_point = Point(intersection_x, intersection_y)
    
    def calculate_distances_and_side(row):
        # Check for missing data
        if pd.isna(row['prev_lon']) or pd.isna(row['prev_lat']):
            dist_to_prev_current = None
        else:
            # Convert points to projected coordinates
            prev_x, prev_y = transformer.transform(row['prev_lon'], row['prev_lat'])
            cur_x, cur_y = transformer.transform(row['lon'], row['lat'])
            
            # Create line segment
            prev_current_line = LineString([(prev_x, prev_y), (cur_x, cur_y)])
            
            # Calculate distance (already in feet)
            dist_to_prev_current = intersection_point.distance(prev_current_line)
        
        if pd.isna(row['next_lon']) or pd.isna(row['next_lat']):
            dist_to_current_next = None
        else:
            # Convert points to projected coordinates
            cur_x, cur_y = transformer.transform(row['lon'], row['lat'])
            next_x, next_y = transformer.transform(row['next_lon'], row['next_lat'])
            
            # Create line segment
            current_next_line = LineString([(cur_x, cur_y), (next_x, next_y)])
            
            # Calculate distance (already in feet)
            dist_to_current_next = intersection_point.distance(current_next_line)
        
        # Determine side
        if dist_to_prev_current is not None and dist_to_current_next is not None:
            if dist_to_prev_current < dist_to_current_next:
                side = "far-side"
            else:
                side = "near-side"
        else:
            side = "unknown"
        
        return pd.Series({
            'dist_to_prev_current': dist_to_prev_current,
            'dist_to_current_next': dist_to_current_next,
            'stop_side': side
        })
    
    # Create a copy to avoid modifying original
    df_copy = df.copy()
    
    # Apply function and join results
    results = df_copy.apply(calculate_distances_and_side, axis=1)
    
    # Assign each column separately
    df_copy['dist_to_prev_current'] = results['dist_to_prev_current']
    df_copy['dist_to_current_next'] = results['dist_to_current_next']
    df_copy['stop_side'] = results['stop_side']
    
    return df_copy

In [32]:
# intersection point
inter_lat = 40.749041  # replace
inter_lon = -73.939690
# bearing from stop -> intersection

filt = filter_within_radius(df, 'lat','lon',inter_lat, inter_lon, 500).drop_duplicates(['route_id','stop_id']).copy()

filt = calculate_bus_stop_side(filt, inter_lat, inter_lon)

filt[['route_id', 'direction_id', 'trip_headsign', 'stop_id', #'stop_sequence',
       'stop_name', 
      #  'lat', 'lon', 
    #    'prev_lat', 'prev_lon', 'next_lat','next_lon',
      #  'dist_prev_cur', 'dist_cur_next', 'intersection_side',
       'dist_to_prev_current','dist_to_current_next','stop_side']].sort_values('stop_id')

,route_id,direction_id,trip_headsign,stop_id,stop_name,dist_to_prev_current,dist_to_current_next,stop_side
5498468,Q63,1,RUSH COURT SQUARE via NORTHERN BL,504413,JACKSON AV/QUEENS PLAZA SOUTH,341.915774,305.316772,near-side
6053118,Q101,1,HUNTERS POINT via STEINWAY ST,504413,JACKSON AV/QUEENS PLAZA SOUTH,341.915774,275.218140,near-side
3596280,Q39,1,GLENDALE,504549,28 ST/QUEENS PLAZA SOUTH,NaN,36.731409,unknown
6097164,Q32,0,JACKSON HEIGHTS,505158,QUEENS PLAZA SOUTH/28 ST,334.971762,345.673461,far-side
5857461,Q60,0,JAMAICA LIRR via QUEENS BL,505158,QUEENS PLAZA SOUTH/28 ST,338.327912,347.030316,far-side
5501864,Q66,0,FLUSHING via NORTHERN BL,550742,28 ST/QUEENS PLAZA SOUTH,NaN,135.205497,unknown
5501557,Q63,0,RUSH FLUSHING via NORTHERN BL,552028,JACKSON AV/ORCHARD ST,354.816037,402.127467,far-side
6053130,Q101,0,STEINWAY via STEINWAY ST,552028,JACKSON AV/ORCHARD ST,347.361230,402.127467,far-side
6087736,Q102,0,ROOSEVELT ISLAND,552028,JACKSON AV/ORCHARD ST,351.475275,402.127467,far-side
6075410,Q69,0,EAST ELMHURST via 21 ST via DITMARS BL,701083,28 ST/42 RD,NaN,54.581973,unknown


### By bearing

In [25]:

PARQ_BASE = "parquet"  # change if needed


def _parquet_path(table_name: str) -> str:
    base = PARQ_BASE.rstrip('/')
    if base.startswith("http"):
        return f"{base}/{table_name}.parquet"
    p = Path(base)
    single = p / f"{table_name}.parquet"
    folder = p / table_name
    if single.exists():
        return single.as_posix()
    # if a folder with parquet files exists, use a wildcard to read all
    if folder.exists() and any(folder.glob("*.parquet")):
        return (folder / "*.parquet").as_posix()
    # fallback to single-file path (DuckDB will error if missing)
    return single.as_posix()


# def parquet_path(table_name: str) -> str:
#     base = PARQ_BASE.rstrip("/")
#     if base.startswith("http"):
#         return f"{base}/{table_name}.parquet"
#     return f"{base}/{table_name}/*.parquet"

def to_sec(hms: str) -> int:
    hh, mm, *rest = hms.split(":")
    ss = int(rest[0]) if rest else 0
    return int(hh) * 3600 + int(mm) * 60 + ss

In [41]:
def buses_by_stop_route_dir_within_radius(
    lon: float,
    lat: float,
    start_time: str,       # "HH:MM" or "HH:MM:SS"
    end_time: str,         # "HH:MM" or "HH:MM:SS"
    day_type: str,         # "Weekday" | "Saturday" | "Sunday"
    radius_ft: int = 250,
    selected_feeds: list[str] | None = None, 
    con: duckdb.DuckDBPyConnection | None = None,
) -> pd.DataFrame:
    """
    Returns one row per (route_id, direction_id, stop_id) within radius,
    with stop name + lat/lon and count of buses in the inclusive time window.
    Handles midnight-spanning windows (e.g., 23:30–00:30).
    """

    # project the query point to EPSG:2263 (NY state plane feet)
    x0, y0 = Transformer.from_crs("EPSG:4326", "EPSG:2263", always_xy=True).transform(lon, lat)
    s, e = to_sec(start_time), to_sec(end_time)

    # define placeholders for feeds selection 
    sel = list(selected_feeds or [])
    if sel:
        values = ",".join(["(?)"] * len(sel))           # -> "(?),(?),(?)"
        chosen_cte = f"chosen_feeds(feed_id) AS (VALUES {values}),"
        feed_pred = "feed_id IN (SELECT feed_id FROM chosen_feeds)"
    else:
        chosen_cte = ""                                  # no CTE
        feed_pred = "TRUE"                               # no filter = all feeds

    # resolve parquet paths (single file or folder wildcard)
    p_dim_stops = _parquet_path('dim_stops')
    p_dim_trips = _parquet_path('dim_trips')
    p_dim_routes = _parquet_path('dim_routes')
    p_calendar_base = _parquet_path('calendar_base')
    p_fact_stop_events = _parquet_path('fact_stop_events')

    sql = f"""
    WITH
    {chosen_cte}
    dim_stops AS (SELECT * FROM read_parquet('{p_dim_stops}')),
    dim_trips  AS (SELECT * FROM read_parquet('{p_dim_trips}')),
    dim_routes AS (SELECT * FROM read_parquet('{p_dim_routes}')),
    calendar_base AS (SELECT * FROM read_parquet('{p_calendar_base}')),
    fact_stop_events AS (SELECT * FROM read_parquet('{p_fact_stop_events}')),
    svcs AS (
      SELECT DISTINCT feed_id, service_id
      FROM calendar_base
      WHERE {feed_pred}
      AND (
        (? = 'Weekday'  AND (monday=1 OR tuesday=1 OR wednesday=1 OR thursday=1 OR friday=1))
        OR (? = 'Saturday' AND saturday=1)
        OR (? = 'Sunday'   AND sunday=1)
        )
    ),
    win AS (SELECT ?::INTEGER AS s, ?::INTEGER AS e),
    near_stops AS (
      SELECT feed_id, stop_id, stop_name, lat, lon
      FROM dim_stops
      WHERE {feed_pred}
      AND ((x2263 - ?)*(x2263 - ?) + (y2263 - ?)*(y2263 - ?)) <= ?*?
    )
    SELECT
      r.feed_id,
      r.route_id,
      t.trip_headsign,
      t.direction_id,
      f.service_id,
      f.trip_id,
      s.stop_id,
      s.stop_name,
      f.stop_sequence,
      s.lat  AS stop_lat,
      s.lon  AS stop_lon,
      COUNT(*) AS buses_scheduled
    FROM fact_stop_events f
    JOIN dim_trips  t ON f.feed_id = t.feed_id AND f.trip_id = t.trip_id
    JOIN dim_routes r ON t.feed_id = r.feed_id AND t.route_id = r.route_id
    JOIN svcs       v ON f.feed_id = v.feed_id AND f.service_id = v.service_id
    JOIN near_stops s ON f.feed_id = s.feed_id AND f.stop_id   = s.stop_id
    CROSS JOIN win
    WHERE
      (
        (SELECT e FROM win) >= (SELECT s FROM win)
        AND f.arrival_sec BETWEEN (SELECT s FROM win) AND (SELECT e FROM win)
      )
      OR
      (
        (SELECT e FROM win) < (SELECT s FROM win)   -- midnight wrap
        AND (f.arrival_sec >= (SELECT s FROM win) OR f.arrival_sec <= (SELECT e FROM win))
      )
    GROUP BY r.feed_id, r.route_id, t.direction_id, t.trip_headsign, s.stop_id, s.stop_name, s.lat, s.lon, f.service_id, f.trip_id, f.stop_sequence,
    ORDER BY s.stop_name, r.feed_id, r.route_id, t.direction_id, f.service_id;
    """

    params = []
    # 1) If feeds are selected, add one param per "(?)" in chosen_feeds CTE
    if sel:
        params += sel

    # 2) Always add the rest: day_type, time window, spatial params
    params += [day_type, day_type, day_type]                # 3 day-type placeholders
    params += [s, e]                                        # window
    params += [x0, x0, y0, y0, int(radius_ft), int(radius_ft)]  # spatial
    df = con.execute(sql, params).fetchdf()

    return df

In [42]:

def buses_by_stop_route_dir_within_radius_with_side(
    lon, lat, start_time, end_time, day_type,
    radius_ft=250, selected_feeds=None, con=None
):
    con = con or duckdb.connect()
    x0, y0 = Transformer.from_crs("EPSG:4326", "EPSG:2263", always_xy=True).transform(lon, lat)
    s, e = to_sec(start_time), to_sec(end_time)

    sel = list(selected_feeds or [])
    if sel:
        values = ",".join(["(?)"] * len(sel))
        chosen_cte = f"chosen_feeds(feed_id) AS (VALUES {values}),"
        feed_pred = "feed_id IN (SELECT feed_id FROM chosen_feeds)"
    else:
        chosen_cte = ""
        feed_pred = "TRUE"

    p_dim_stops = parquet_path("dim_stops")
    p_dim_trips = parquet_path("dim_trips")
    p_dim_routes = parquet_path("dim_routes")
    p_calendar_base = parquet_path("calendar_base")
    p_fact_stop_events = parquet_path("fact_stop_events")

    sql = f"""
    WITH
    {chosen_cte}
    dim_stops AS (SELECT * FROM read_parquet('{p_dim_stops}')),
    dim_trips  AS (SELECT * FROM read_parquet('{p_dim_trips}')),
    dim_routes AS (SELECT * FROM read_parquet('{p_dim_routes}')),
    calendar_base AS (SELECT * FROM read_parquet('{p_calendar_base}')),
    fact_stop_events AS (SELECT * FROM read_parquet('{p_fact_stop_events}')),
    svcs AS (
      SELECT DISTINCT feed_id, service_id
      FROM calendar_base
      WHERE {feed_pred}
      AND (
        (? = 'Weekday'  AND (monday=1 OR tuesday=1 OR wednesday=1 OR thursday=1 OR friday=1))
        OR (? = 'Saturday' AND saturday=1)
        OR (? = 'Sunday'   AND sunday=1)
        )
    ),
    win AS (SELECT ?::INTEGER AS s, ?::INTEGER AS e),
    near_stops AS (
      SELECT feed_id, stop_id, stop_name, lat, lon
      FROM dim_stops
      WHERE {feed_pred}
      AND ((x2263 - ?)*(x2263 - ?) + (y2263 - ?)*(y2263 - ?)) <= ?*?
    )
    SELECT
      r.feed_id,
      r.route_id,
      t.trip_headsign,
      t.direction_id,
      f.service_id,
      s.stop_id,
      s.stop_name,
      s.lat  AS stop_lat,
      s.lon  AS stop_lon,
      COUNT(*) AS buses_scheduled
    FROM fact_stop_events f
    JOIN dim_trips  t ON f.feed_id = t.feed_id AND f.trip_id = t.trip_id
    JOIN dim_routes r ON t.feed_id = r.feed_id AND t.route_id = r.route_id
    JOIN svcs       v ON f.feed_id = v.feed_id AND f.service_id = v.service_id
    JOIN near_stops s ON f.feed_id = s.feed_id AND f.stop_id   = s.stop_id
    CROSS JOIN win
    WHERE
      (
        (SELECT e FROM win) >= (SELECT s FROM win)
        AND f.arrival_sec BETWEEN (SELECT s FROM win) AND (SELECT e FROM win)
      )
      OR
      (
        (SELECT e FROM win) < (SELECT s FROM win)
        AND (f.arrival_sec >= (SELECT s FROM win) OR f.arrival_sec <= (SELECT e FROM win))
      )
    GROUP BY r.feed_id, r.route_id, t.direction_id, t.trip_headsign, s.stop_id, s.stop_name, s.lat, s.lon, f.service_id
    ORDER BY s.stop_name, r.feed_id, r.route_id, t.direction_id, f.service_id;
    """

    params = []
    if sel:
        params += sel
    params += [day_type, day_type, day_type]
    params += [s, e]
    params += [x0, x0, y0, y0, int(radius_ft), int(radius_ft)]
    df = con.execute(sql, params).fetchdf()

    if df.empty:
        return df

    # ---- bearings per stop (from stop_sequence)
    bearings_sql = f"""
    WITH
    {chosen_cte}
    dim_stops AS (SELECT * FROM read_parquet('{p_dim_stops}')),
    calendar_base AS (SELECT * FROM read_parquet('{p_calendar_base}')),
    fact_stop_events AS (SELECT * FROM read_parquet('{p_fact_stop_events}')),
    svcs AS (
      SELECT DISTINCT feed_id, service_id
      FROM calendar_base
      WHERE {feed_pred}
      AND (
        (? = 'Weekday'  AND (monday=1 OR tuesday=1 OR wednesday=1 OR thursday=1 OR friday=1))
        OR (? = 'Saturday' AND saturday=1)
        OR (? = 'Sunday'   AND sunday=1)
        )
    ),
    near_stops AS (
      SELECT feed_id, stop_id
      FROM dim_stops
      WHERE {feed_pred}
      AND ((x2263 - ?)*(x2263 - ?) + (y2263 - ?)*(y2263 - ?)) <= ?*?
    ),
    events AS (
      SELECT
        f.feed_id, f.route_id, f.direction_id, f.service_id, f.trip_id, f.stop_id, f.stop_sequence,
        LAG(f.stop_id)  OVER (PARTITION BY f.feed_id, f.trip_id ORDER BY f.stop_sequence) AS prev_stop_id,
        LEAD(f.stop_id) OVER (PARTITION BY f.feed_id, f.trip_id ORDER BY f.stop_sequence) AS next_stop_id
      FROM fact_stop_events f
      JOIN svcs v ON f.feed_id = v.feed_id AND f.service_id = v.service_id
      WHERE {feed_pred}
    ),
    bearing_base AS (
      SELECT
        e.feed_id, e.route_id, e.direction_id, e.service_id, e.stop_id,
        COALESCE(p.lat, c.lat) AS a_lat,
        COALESCE(p.lon, c.lon) AS a_lon,
        COALESCE(n.lat, c.lat) AS b_lat,
        COALESCE(n.lon, c.lon) AS b_lon
      FROM events e
      JOIN near_stops ns ON e.feed_id = ns.feed_id AND e.stop_id = ns.stop_id
      LEFT JOIN dim_stops c ON e.feed_id = c.feed_id AND e.stop_id = c.stop_id
      LEFT JOIN dim_stops p ON e.feed_id = p.feed_id AND e.prev_stop_id = p.stop_id
      LEFT JOIN dim_stops n ON e.feed_id = n.feed_id AND e.next_stop_id = n.stop_id
    ),
    bearing_events AS (
      SELECT *,
        atan2(
          sin(radians(b_lon - a_lon)) * cos(radians(b_lat)),
          cos(radians(a_lat)) * sin(radians(b_lat))
            - sin(radians(a_lat)) * cos(radians(b_lat)) * cos(radians(b_lon - a_lon))
        ) AS bearing_rad
      FROM bearing_base
      WHERE a_lat IS NOT NULL AND b_lat IS NOT NULL
        AND (a_lat != b_lat OR a_lon != b_lon)
    )
    SELECT
      feed_id, route_id, direction_id, service_id, stop_id,
      atan2(AVG(sin(bearing_rad)), AVG(cos(bearing_rad))) AS mean_bearing_rad
    FROM bearing_events
    GROUP BY feed_id, route_id, direction_id, service_id, stop_id;
    """

    bparams = []
    if sel:
        bparams += sel
    bparams += [day_type, day_type, day_type]
    bparams += [x0, x0, y0, y0, int(radius_ft), int(radius_ft)]
    bearings = con.execute(bearings_sql, bparams).fetchdf()

    if not bearings.empty:
        bearings["travel_bearing_deg"] = (np.degrees(bearings["mean_bearing_rad"]) + 360.0) % 360.0
        dirs = np.array(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])
        bearings["travel_dir_cardinal"] = dirs[((bearings["travel_bearing_deg"] + 22.5) // 45).astype(int) % 8]
        df = df.merge(
            bearings[["feed_id","route_id","direction_id","service_id","stop_id","travel_bearing_deg","travel_dir_cardinal"]],
            on=["feed_id","route_id","direction_id","service_id","stop_id"],
            how="left",
        )
    else:
        df["travel_bearing_deg"] = np.nan
        df["travel_dir_cardinal"] = None

    df["bearing_to_intersection_deg"] = df.apply(
        lambda r: _bearing_deg(r["stop_lat"], r["stop_lon"], lat, lon), axis=1
    )

    def _side(row):
        if pd.isna(row.get("travel_bearing_deg")):
            return "unknown"
        diff = abs((row["travel_bearing_deg"] - row["bearing_to_intersection_deg"] + 180) % 360 - 180)
        return "near_side" if diff <= 90 else "far_side"

    df["intersection_side"] = df.apply(_side, axis=1)
    return df


In [43]:
df = buses_by_stop_route_dir_within_radius(
    lon=-73.9855, lat=40.7580,
    start_time="07:45:00", end_time="08:45:00",
    day_type="Weekday",
    radius_ft=250,
    selected_feeds=None,
    con=con,
)
df.head()

,feed_id,route_id,trip_headsign,direction_id,service_id,trip_id,stop_id,stop_name,stop_sequence,stop_lat,stop_lon,buses_scheduled
0,mta-nyct-bus-manhattan,M104,41 ST via BROADWAY/7 AV,1,MV_A6-Weekday-SDon,MV_A6-Weekday-SDon-043500_M104_103,405297,7 AV/W 44 ST,40,40.757476,-73.985927,1
1,mta-nyct-bus-manhattan,M104,41 ST via BROADWAY/7 AV,1,MV_A6-Weekday-SDon,MV_A6-Weekday-SDon-044500_M104_110,405297,7 AV/W 44 ST,40,40.757476,-73.985927,1
2,mta-nyct-bus-manhattan,M104,41 ST via BROADWAY/7 AV,1,MV_A6-Weekday-SDon,MV_A6-Weekday-SDon-042300_M104_109,405297,7 AV/W 44 ST,40,40.757476,-73.985927,1
3,mta-nyct-bus-manhattan,M104,41 ST via BROADWAY/7 AV,1,MV_A6-Weekday-SDon,MV_A6-Weekday-SDon-045500_M104_104,405297,7 AV/W 44 ST,40,40.757476,-73.985927,1
4,mta-nyct-bus-manhattan,M104,41 ST via BROADWAY/7 AV,1,MV_H6-Weekday,MV_H6-Weekday-043200_M104_103,405297,7 AV/W 44 ST,40,40.757476,-73.985927,1


In [139]:
# intersection point
inter_lat = 40.769906  # replace
inter_lon = -73.927807
# bearing from stop -> intersection

filt = df[(df['route_id'].isin(['Q18','B62','Q69','Q100'])) 
          & (df['lat'].between(40.74,40.775))
          & (df['lon'].between(-73.930,-73.926))
          ][['route_id', 'direction_id','trip_headsign', 'stop_id', 'stop_sequence',
       'stop_name', 'lat', 'lon',
       'prev_lat', 'prev_lon', 'next_lat', 'next_lon', 'bearing_prev_cur',
       'bearing_cur_next', 'bearing_prev_next',]].copy()
filt["bearing_to_int"] = bearing_deg(filt["lat"], filt["lon"], inter_lat, inter_lon)
filt["bearing_to_int_2"] = bearing_deg(inter_lat, inter_lon,filt["lat"], filt["lon"])
filt.drop_duplicates(['route_id','stop_id'])

,route_id,direction_id,trip_headsign,stop_id,stop_sequence,stop_name,lat,lon,prev_lat,prev_lon,next_lat,next_lon,bearing_prev_cur,bearing_cur_next,bearing_prev_next,bearing_to_int,bearing_to_int_2
3582458,B62,1,DOWNTOWN BROOKLYN,550689,3,21 ST/30 AV,40.769676,-73.928113,40.772756,-73.932703,40.765371,-73.931688,131.540953,212.168542,174.057126,45.216536,225.216736
3582754,B62,0,ASTORIA,550523,50,30 AV/21 ST,40.770126,-73.928178,40.765429,-73.931349,40.772953,-73.932424,27.079820,311.321946,353.824592,128.060688,308.060930
6073727,Q69,1,QUEENS PLAZA via DITMARS BL via 21 ST,550693,16,21 ST/ASTORIA BLVD,40.772185,-73.926139,40.774496,-73.924252,40.769815,-73.928066,211.731484,211.623999,211.677719,208.999546,28.998456
6073728,Q69,1,QUEENS PLAZA via DITMARS BL via 21 ST,550689,17,21 ST/30 AV,40.769815,-73.928066,40.772185,-73.926139,40.767282,-73.930056,211.623999,210.753274,211.176810,65.111964,245.112133
6075420,Q69,0,EAST ELMHURST via 21 ST via DITMARS BL,550688,11,21 ST/30 AV,40.770106,-73.927583,40.766756,-73.930170,40.772396,-73.925725,30.320766,31.568317,30.830543,220.305264,40.305118
6080930,Q100,1,LIMITED QUEENS PLAZA,550693,7,21 ST/ASTORIA BLVD,40.772185,-73.926139,40.776791,-73.921467,40.769815,-73.928066,217.530732,211.623999,215.618949,208.999546,28.998456
6080931,Q100,1,LIMITED QUEENS PLAZA,550689,8,21 ST/30 AV,40.769815,-73.928066,40.772185,-73.926139,40.765227,-73.931869,211.623999,212.121073,211.953195,65.111964,245.112133
6080974,Q100,0,LIMITED STEINWAY,550688,4,21 ST/30 AV,40.770106,-73.927583,40.765445,-73.931388,40.770106,-73.927583,31.726084,0.000000,31.726084,220.305264,40.305118
6088567,Q18,0,MASPETH,550520,5,30 AV/14 ST,40.770702,-73.929828,40.772927,-73.932745,40.769704,-73.927436,135.204605,118.850073,128.714525,117.476805,297.478125
6088568,Q18,0,MASPETH,550522,6,30 AV/21 ST,40.769704,-73.927436,40.770702,-73.929828,40.768237,-73.924500,118.850073,123.413796,121.418425,305.713634,125.713392


In [142]:
def ang_diff(a, b):
    # smallest signed angle difference (degrees)
    return (a - b + 180) % 360 - 180

# pick the bearing that points most toward the intersection
# (smallest absolute angle difference)
cand_cols = ["bearing_prev_cur", "bearing_cur_next"]
diff_prev = np.abs(ang_diff(filt["bearing_prev_cur"], filt["bearing_to_int"]))
diff_next = np.abs(ang_diff(filt["bearing_cur_next"], filt["bearing_to_int"]))

filt["travel_bearing_at_int"] = np.where(
    diff_prev <= diff_next, filt["bearing_prev_cur"], filt["bearing_cur_next"]
)

filt["intersection_side"] = np.where(
    diff_next < diff_prev, "near_side", "far_side"
)

diff_prev = np.abs(ang_diff(filt["bearing_prev_cur"], filt["bearing_to_int_2"]))
diff_next = np.abs(ang_diff(filt["bearing_cur_next"], filt["bearing_to_int_2"]))

filt["travel_bearing_at_int_2"] = np.where(
    diff_prev <= diff_next, filt["bearing_prev_cur"], filt["bearing_cur_next"]
)

filt["intersection_side_2"] = np.where(
    diff_next < diff_prev, "near_side", "far_side"
)

filt[['route_id', 'direction_id','trip_headsign', 'stop_id', 'stop_name','intersection_side', 'intersection_side_2',
       'lat', 'lon', 'prev_lat', 'prev_lon', 'next_lat', 'next_lon',
       'bearing_prev_cur', 'bearing_cur_next', 'bearing_prev_next',
       'bearing_to_int', 'bearing_to_int_2', 'travel_bearing_at_int',
       'travel_bearing_at_int_2', ]].drop_duplicates(['route_id','stop_id','direction_id']).sort_values(['stop_id','route_id'])

,route_id,direction_id,trip_headsign,stop_id,stop_name,intersection_side,intersection_side_2,lat,lon,prev_lat,prev_lon,next_lat,next_lon,bearing_prev_cur,bearing_cur_next,bearing_prev_next,bearing_to_int,bearing_to_int_2,travel_bearing_at_int,travel_bearing_at_int_2
6088567,Q18,0,MASPETH,550520,30 AV/14 ST,near_side,far_side,40.770702,-73.929828,40.772927,-73.932745,40.769704,-73.927436,135.204605,118.850073,128.714525,117.476805,297.478125,118.850073,135.204605
6088568,Q18,0,MASPETH,550522,30 AV/21 ST,far_side,near_side,40.769704,-73.927436,40.770702,-73.929828,40.768237,-73.924500,118.850073,123.413796,121.418425,305.713634,125.713392,118.850073,123.413796
3582754,B62,0,ASTORIA,550523,30 AV/21 ST,far_side,near_side,40.770126,-73.928178,40.765429,-73.931349,40.772953,-73.932424,27.079820,311.321946,353.824592,128.060688,308.060930,27.079820,311.321946
6090877,Q18,1,ASTORIA,550523,30 AV/21 ST,near_side,far_side,40.770058,-73.927899,40.768754,-73.925144,40.771497,-73.931046,302.005267,301.123836,301.538314,155.373800,335.373860,301.123836,302.005267
6080974,Q100,0,LIMITED STEINWAY,550688,21 ST/30 AV,near_side,far_side,40.770106,-73.927583,40.765445,-73.931388,40.770106,-73.927583,31.726084,0.000000,31.726084,220.305264,40.305118,0.000000,31.726084
6075420,Q69,0,EAST ELMHURST via 21 ST via DITMARS BL,550688,21 ST/30 AV,far_side,near_side,40.770106,-73.927583,40.766756,-73.930170,40.772396,-73.925725,30.320766,31.568317,30.830543,220.305264,40.305118,30.320766,31.568317
3582458,B62,1,DOWNTOWN BROOKLYN,550689,21 ST/30 AV,far_side,near_side,40.769676,-73.928113,40.772756,-73.932703,40.765371,-73.931688,131.540953,212.168542,174.057126,45.216536,225.216736,131.540953,212.168542
6080931,Q100,1,LIMITED QUEENS PLAZA,550689,21 ST/30 AV,far_side,near_side,40.769815,-73.928066,40.772185,-73.926139,40.765227,-73.931869,211.623999,212.121073,211.953195,65.111964,245.112133,211.623999,212.121073
6073728,Q69,1,QUEENS PLAZA via DITMARS BL via 21 ST,550689,21 ST/30 AV,near_side,far_side,40.769815,-73.928066,40.772185,-73.926139,40.767282,-73.930056,211.623999,210.753274,211.176810,65.111964,245.112133,210.753274,211.623999
6080930,Q100,1,LIMITED QUEENS PLAZA,550693,21 ST/ASTORIA BLVD,near_side,far_side,40.772185,-73.926139,40.776791,-73.921467,40.769815,-73.928066,217.530732,211.623999,215.618949,208.999546,28.998456,211.623999,217.530732
